# **Model Call And Tool Run Limit Middleware**

## **What's Covered?**
1. Model Call Limit Middleware
2. Tool Run Limit Middleware

## **Model Call Limit Middleware**

Limit the number of model calls to prevent infinite loops or excessive costs. 

Model call limit is useful for the following:
- Preventing runaway agents from making too many API calls.
- Enforcing cost controls on production deployments.
- Testing agent behavior within specific call budgets.

This middleware monitors the number of model calls made during agent execution and can terminate the agent when specified limits are reached. It supports both **thread-level** and **run-level** call counting with configurable exit behaviors.
- **Thread-level (i.e. Conversation Level)**: The middleware tracks the number of model calls and persists call count across multiple runs (invocations) of the agent.
- **Run-level (i.e. Single Invocation)**: The middleware tracks the number of model calls made during a single run (invocation) of the agent.

**Configuration options:**
- thread_limit: Maximum model calls across all runs in a thread. Defaults to no limit. For thread_limit, it is mandatory to provide a `checkpointer`
- run_limit: Maximum model calls per single invocation. Defaults to no limit.
- exit_behavior: Behavior when limit is reached. Options: `end` (graceful termination) or `error` (raise exception). Default to `end`.

In [1]:
from langchain_core.tools import tool

@tool
def multiply_tool(a: int, b: int) -> int:
    """This tool take two integer variables in the input and returns the product"""
    return a * b

In [2]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

# Create middleware with limits
call_tracker = ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")

agent = create_agent(
    model=chat_model,
    tools=[multiply_tool],
    middleware=[call_tracker]
)

In [4]:
response = agent.invoke({"messages": "help me multiply 3 and 4"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_16414aeb-747e-4167-821a-9d18012409a4)
 Call ID: fc_16414aeb-747e-4167-821a-9d18012409a4
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================

Sure! 3 × 4 = 12.


In [11]:
response = agent.invoke({"messages": "help me multiply 3 and 4 and 5"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4 and 5
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_8fb209ff-321f-443f-a56f-6b99a57dd804)
 Call ID: fc_8fb209ff-321f-443f-a56f-6b99a57dd804
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_b435fb1e-9bc8-423a-a863-9156d583acff)
 Call ID: fc_b435fb1e-9bc8-423a-a863-9156d583acff
  Args:
    a: 12
    b: 5
================================= Tool Message =================================
Name: multiply_tool

60
================================== Ai Message ==================================

Model call limits exceeded: run limit (2/2)


## **Tool Run Limit Middleware**

- tool_name: str | None = None
- thread_limit: int | None = None
- run_limit: int | None = None
- exit_behavior:
    - 'continue': Block exceeded tools, let execution continue (default), agent resumes
    - 'error': raise informative error message
    - 'end': Stop immediately with a ToolMessage + AI message for the single tool call that exceeded the limit (raises NotImplementedError if there are other pending tool calls (due to parallel tool calling).

In [5]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_items(query: str) -> str:
    """Search for items in the store.
    
    Args:
        query: Search query to find items.
        
    Returns:
        List of matching items with IDs and prices.
    """
    return """Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5"""


@tool(parse_docstring=True)
def purchase_item(item_id: str) -> str:
    """Purchase an item by its ID.
    
    Args:
        item_id: The unique identifier of the item to purchase.
        
    Returns:
        Purchase confirmation.
    """
    # Simulate purchase
    return f"✓ Successfully purchased item {item_id}"

In [6]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)

chat_model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

PROMPT = """
You are a shopping assistant. Help users search for and purchase items.
Use the search_items tool to find items and the purchase_item tool to purchase items.
Make sure to use the ratings of the items to make the best purchase decision.
"""

agent = create_agent(
    model=chat_model,
    tools=[search_items, purchase_item],
    system_prompt=PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="purchase_item",
            run_limit=1,  # Max 1 purchase per conversation turn
            thread_limit=2,  # Max 2 purchases total across all conversation turns
        )
    ],
)

In [8]:
response = agent.invoke({"messages": "find the headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

find the headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_faf716bc-7c32-4530-959a-f43c19ea9339)
 Call ID: fc_faf716bc-7c32-4530-959a-f43c19ea9339
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================

Here are the headphones under $300:

| ID | Product | Price | Rating |
|----|---------|-------|--------|
| **hdpn-002** | Bose QuietComfort 45 | **$279.00** | **4.7 / 5** |
| hdpn-001 | Sony WH‑1000XM5 | $299.99 | 4.5 / 5 |

The Bose QuietComfort 45 has the higher

In [9]:
response = agent.invoke({"messages": "purchase a Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase a Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_3e437333-3d77-43b0-86eb-9cba62f8c285)
 Call ID: fc_3e437333-3d77-43b0-86eb-9cba62f8c285
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_cf2b092c-6cab-4081-b324-a791f3ca5992)
 Call ID: fc_cf2b092c-6cab-4081-b324-a791f3ca5992
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchase_item

✓ Successfull

In [10]:
response = agent.invoke({"messages": "purchase me 2 pairs of Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase me 2 pairs of Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_3f36601e-27d9-466e-8925-157b199509da)
 Call ID: fc_3f36601e-27d9-466e-8925-157b199509da
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_87df53b3-f5ef-48da-9588-78f07a7a87d7)
 Call ID: fc_87df53b3-f5ef-48da-9588-78f07a7a87d7
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchase_item

✓

In [12]:
response = agent.invoke({"messages": "purchase me two best pairs of headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase me two best pairs of headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_e9a9804f-d19b-4f73-bd3f-221b38fbd9b7)
 Call ID: fc_e9a9804f-d19b-4f73-bd3f-221b38fbd9b7
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_58490003-ed71-42df-8f7d-9c12c2b3110e)
 Call ID: fc_58490003-ed71-42df-8f7d-9c12c2b3110e
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchas